In [1]:
from pathlib import Path

DATASET_PATH = Path(r"C:\Users\HP\Desktop\RHD_Project\data\PCG\extracted\Rhd detection using phonocardiogram\Rhd detection using phonocardiogram\Dataset")

print("Dataset exists:", DATASET_PATH.exists())
print("Dataset path:", DATASET_PATH)

Dataset exists: True
Dataset path: C:\Users\HP\Desktop\RHD_Project\data\PCG\extracted\Rhd detection using phonocardiogram\Rhd detection using phonocardiogram\Dataset


In [2]:
# Check files inside the dataset

files = list(DATASET_PATH.iterdir())

print("Total files:", len(files))

print("\nFirst 20 files:")
for file in files[:20]:
    print(file.name)

Total files: 1230

First 20 files:
a0001.dat
a0001.hea
a0001.wav
a0002.dat
a0002.hea
a0002.wav
a0003.dat
a0003.hea
a0003.wav
a0004.dat
a0004.hea
a0004.wav
a0005.dat
a0005.hea
a0005.wav
a0006.dat
a0006.hea
a0006.wav
a0007.dat
a0007.hea


In [3]:
from collections import Counter

extensions = Counter(file.suffix.lower() for file in files)

print("File type counts:\n")

for ext, count in sorted(extensions.items()):
    print(f"{ext or '[no extension]'} : {count}")

File type counts:

[no extension] : 6
.csv : 1
.dat : 405
.hea : 409
.wav : 409


In [4]:
# Find CSV files

csv_files = [file for file in files if file.suffix.lower() == ".csv"]

for file in csv_files:
    print("CSV file:", file.name)

CSV file: REFERENCE.csv


In [6]:
import csv

REFERENCE_PATH = DATASET_PATH / "REFERENCE.csv"

with open(REFERENCE_PATH, "r") as f:
    reader = csv.reader(f)
    rows = list(reader)

print("Total rows:", len(rows))

print("\nFirst 10 rows:")
for row in rows[:10]:
    print(row)

Total rows: 409

First 10 rows:
['a0001', '1']
['a0002', '1']
['a0003', '1']
['a0004', '1']
['a0005', '1']
['a0006', '1']
['a0007', '-1']
['a0008', '1']
['a0009', '-1']
['a0010', '1']


In [7]:
from collections import Counter

labels = [row[1] for row in rows]

label_counts = Counter(labels)

print("Label distribution:")
for label, count in label_counts.items():
    print(f"Label {label}: {count}")

Label distribution:
Label 1: 292
Label -1: 117


In [8]:
# Check RECORDS-normal and RECORDS-abnormal

normal_file = DATASET_PATH / "RECORDS-normal"
abnormal_file = DATASET_PATH / "RECORDS-abnormal"

with open(normal_file, "r") as f:
    normal_records = [line.strip() for line in f if line.strip()]

with open(abnormal_file, "r") as f:
    abnormal_records = [line.strip() for line in f if line.strip()]

print("Normal recordings:", len(normal_records))
print("Abnormal recordings:", len(abnormal_records))

print("\nFirst 10 normal records:")
print(normal_records[:10])

print("\nFirst 10 abnormal records:")
print(abnormal_records[:10])

Normal recordings: 117
Abnormal recordings: 292

First 10 normal records:
['a0007', 'a0009', 'a0011', 'a0012', 'a0016', 'a0019', 'a0025', 'a0027', 'a0028', 'a0029']

First 10 abnormal records:
['a0001', 'a0002', 'a0003', 'a0004', 'a0005', 'a0006', 'a0008', 'a0010', 'a0013', 'a0014']


In [9]:
import wave

wav_files = sorted(DATASET_PATH.glob("*.wav"))

print("Number of WAV files:", len(wav_files))

# Inspect the first WAV file
sample_wav = wav_files[0]

with wave.open(str(sample_wav), "rb") as audio:
    print("\nSample file:", sample_wav.name)
    print("Channels:", audio.getnchannels())
    print("Sample rate:", audio.getframerate())
    print("Sample width:", audio.getsampwidth(), "bytes")
    print("Number of frames:", audio.getnframes())
    print("Duration:", round(audio.getnframes() / audio.getframerate(), 2), "seconds")

Number of WAV files: 409

Sample file: a0001.wav
Channels: 1
Sample rate: 2000
Sample width: 2 bytes
Number of frames: 71332
Duration: 35.67 seconds


In [10]:
import wave
from collections import Counter

audio_info = []

for wav_file in wav_files:
    with wave.open(str(wav_file), "rb") as audio:
        channels = audio.getnchannels()
        sample_rate = audio.getframerate()
        sample_width = audio.getsampwidth()
        frames = audio.getnframes()
        duration = frames / sample_rate

        audio_info.append({
            "file": wav_file.name,
            "channels": channels,
            "sample_rate": sample_rate,
            "sample_width": sample_width,
            "frames": frames,
            "duration": duration
        })

print("Total recordings checked:", len(audio_info))

print("\nChannels:", Counter(x["channels"] for x in audio_info))
print("Sample rates:", Counter(x["sample_rate"] for x in audio_info))
print("Sample widths:", Counter(x["sample_width"] for x in audio_info))

durations = [x["duration"] for x in audio_info]

print("\nDuration statistics:")
print("Minimum:", round(min(durations), 2), "seconds")
print("Maximum:", round(max(durations), 2), "seconds")
print("Average:", round(sum(durations) / len(durations), 2), "seconds")

Total recordings checked: 409

Channels: Counter({1: 409})
Sample rates: Counter({2000: 409})
Sample widths: Counter({2: 409})

Duration statistics:
Minimum: 9.27 seconds
Maximum: 36.5 seconds
Average: 32.59 seconds


In [11]:
# Find shortest and longest recordings

shortest = min(audio_info, key=lambda x: x["duration"])
longest = max(audio_info, key=lambda x: x["duration"])

print("Shortest recording:")
print(shortest)

print("\nLongest recording:")
print(longest)

Shortest recording:
{'file': 'a0238.wav', 'channels': 1, 'sample_rate': 2000, 'sample_width': 2, 'frames': 18530, 'duration': 9.265}

Longest recording:
{'file': 'a0324.wav', 'channels': 1, 'sample_rate': 2000, 'sample_width': 2, 'frames': 73004, 'duration': 36.502}


In [13]:
# Count recordings in different duration ranges

ranges = {
    "Under 10 sec": 0,
    "10–20 sec": 0,
    "20–30 sec": 0,
    "30–35 sec": 0,
    "35–40 sec": 0
}

for duration in durations:
    if duration < 10:
        ranges["Under 10 sec"] += 1
    elif duration < 20:
        ranges["10–20 sec"] += 1
    elif duration < 30:
        ranges["20–30 sec"] += 1
    elif duration < 35:
        ranges["30–35 sec"] += 1
    else:
        ranges["35–40 sec"] += 1

print("Duration distribution:\n")

for duration_range, count in ranges.items():
    print(f"{duration_range}: {count}")

Duration distribution:

Under 10 sec: 1
10–20 sec: 6
20–30 sec: 49
30–35 sec: 98
35–40 sec: 255


In [14]:
import wave
import struct

sample_file = DATASET_PATH / "a0001.wav"

with wave.open(str(sample_file), "rb") as audio:
    sample_rate = audio.getframerate()
    n_frames = audio.getnframes()
    raw_data = audio.readframes(n_frames)

# Convert 16-bit audio bytes into integer samples
samples = struct.unpack("<" + "h" * (len(raw_data) // 2), raw_data)

print("File:", sample_file.name)
print("Number of samples:", len(samples))
print("Sample rate:", sample_rate)

print("\nFirst 20 samples:")
print(samples[:20])

print("\nSignal statistics:")
print("Minimum:", min(samples))
print("Maximum:", max(samples))
print("Mean:", round(sum(samples) / len(samples), 2))

File: a0001.wav
Number of samples: 71332
Sample rate: 2000

First 20 samples:
(-69, -135, -104, -25, -63, -3, 30, -22, 26, 77, 144, 139, 157, 251, 211, 300, 256, 317, 417, 419)

Signal statistics:
Minimum: -4503
Maximum: 6880
Mean: -0.8


In [15]:
try:
    import matplotlib.pyplot as plt
    print("Matplotlib is available.")
except ModuleNotFoundError:
    print("Matplotlib is NOT installed.")

Matplotlib is NOT installed.
